# Disciplinary Citation Flow Analysis

**Research Questions addressed in this notebook:**
1. What are the scholarly disciplines that either cite or are cited by the IRIS publications included in OpenCitations for a given institution?
2. Do different institutions show different disciplinary citation patterns?

**Institutions covered:** The abbreviated form will be used in this notebook.
- UNIBO: University of Bologna
- UNIMI: University of Milan
- UNIPD: University of Padua
- UNITO: University of Turin
- UPO: University of Eastern Piedmont
- SNS: Scuola Normale Superiore

**Scholarly Disciplines:**
Disciplines are based on the Library of Congress Main Classifications, with addition of "Multidisciplinary" and "others" as defined and mapped in the preceding preprocessing steps.
    <details>
    <summary>   <b> Click to view all the disciplines</b></summary>  
List of disciplines (Mapped with Library of Congress Main Classifications / alphabetical order). Some labels will be abbreviated in this notebook for space efficiency.
| No. | Label (Discipline) | Abbreviation |
|---|---|---|
| 1 | AUXILIARY SCIENCES OF HISTORY | AUX.SCI.HISTORY |
| 2 | BIBLIOGRAPHY. LIBRARY SCIENCE. INFORMATION RESOURCES | BIB./LIB.SCI. |
| 3 | EDUCATION | - |
| 4 | FINE ARTS | - |
| 5 | GENERAL WORKS | - |
| 6 | GEOGRAPHY. ANTHROPOLOGY. RECREATION | GEOGR./ANTHROP. |
| 7 | LAW | - |
| 8 | LANGUAGE AND LITERATURE | LANG. & LIT. |
| 9 | MEDICINE | - |
| 10 | MILITARY SCIENCE | MILITARY SCI. |
| 11 | MUSIC AND BOOKS ON MUSIC | MUSIC |
| 12 | NAVAL SCIENCE | NAVAL SCI. |
| 13 | PHILOSOPHY. PSYCHOLOGY. RELIGION | PHILOS./PSYCH./REL. |
| 14 | POLITICAL SCIENCE | POLITICAL SCI. |
| 15 | SCIENCE | - |
| 16 | SOCIAL SCIENCES | SOCIAL SCI. |
| 17 | TECHNOLOGY | - |
| 18 | Multidisciplinary | MULTI. |
| 19 | Others | OTHER |

</details>

**Dataset:** For each institution, two CSV files are used.
- `[INSTITUTION]_profile_output.csv` — Counts of disciplines that are "citing IRIS publications" and "cited by IRIS publications"
- `[INSTITUTION]_agg_output.csv` — Aggregated counts of all citing–cited discipline pairs

---

### Structure of this notebook

#### 1. Data Loading and Setup
- Library installation and imports
- Data paths and global constants (institutions, color maps, abbreviations)

#### 2. Overall Disciplinary Distribution Analysis
Overview of how citing and cited disciplines are distributed across institutions.

- **Butterfly Chart** — mirrored horizontal bars comparing citing vs. cited volume per discipline
- **Grouped Bar Chart** — same data as vertical grouped bars for direct comparison
- **100% Stacked Bar Chart** — normalised view highlighting disciplinary composition
- **100% Stacked Bar Chart by Volume Range** — separate panels for different volume ranges; high-range (Medicine, Science), mid-range, and low-range disciplines to address skewed distribution

#### 3. Disciplinary Citation Flow Analysis
Examination of directed citation flows between discipline pairs.

- **Citing–Cited Pair Ranking** — top citing–cited discipline pairs per institution
- **Self vs. Cross Citation by Discipline** — degree of within-discipline vs. cross-discipline citation
- **Self vs. Cross Citation by Institution × Discipline Range** — aggregated self/cross patterns stratified by range group
- **Proportion Sunburst Chart** — interactive proportional overview of citation flows
- **Sankey Diagram (static)** — directional flow and volume for top N pairs
- **Sankey Diagram (interactive)** — filterable version with institution/discipline dropdowns

#### 4. Cross-Institutional Comparison
Comparison of disciplinary citation patterns across the six institutions.

- **Relative Disciplinary Specialisation Heatmap** — institution × discipline deviation from the overall mean

#### 5. Discussion
*(to be completed)*

---

## 1. Data Loading and Setup

In [1]:
# Install required libraries
%pip install plotly
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import plotly.colors

# ── Data paths ────────────────────────────────────────────────────────────────
Root = "https://raw.githubusercontent.com/open-sci/2025-2026/refs/heads/1b/bloom/disciplinary_flow/step4_output/"

subject_profiles = {
    "UNIBO": Root + "unibo_profile_output.csv",
    "UNITO": Root + "unito_profile_output.csv",
    "UNIMI": Root + "unimi_profile_output.csv",
    "UNIPD": Root + "unipd_profile_output.csv",
    "UPO":   Root + "upo_profile_output.csv",
    "SNS":   Root + "sns_profile_output.csv",
}

agg_csvs = {
    "UNIBO": Root + "unibo_agg_output.csv",
    "UNITO": Root + "unito_agg_output.csv",
    "UNIMI": Root + "unimi_agg_output.csv",
    "UNIPD": Root + "unipd_agg_output.csv",
    "UPO":   Root + "upo_agg_output.csv",
    "SNS":   Root + "sns_agg_output.csv",
}

# ── Global constants (shared across all analyses) ─────────────────────────────
INSTITUTIONS = ["UNIBO", "UNITO", "UNIMI", "UNIPD", "UPO", "SNS"]

# Abbreviation map for long discipline names
DISC_SHORT = {
    "GEOGRAPHY. ANTHROPOLOGY. RECREATION":                   "GEO./ANTHROP.",
    "PHILOSOPHY. PSYCHOLOGY. RELIGION":                      "PHIL./PSYCH./REL.",
    "BIBLIOGRAPHY. LIBRARY SCIENCE. INFORMATION RESOURCES":  "BIB./LIB.SCI.",
    "AUXILIARY SCIENCES OF HISTORY":                         "AUX.SCI.HISTORY",
    "SOCIAL SCIENCES":                                       "SOCI.SCI.",
    "POLITICAL SCIENCE":                                     "POLI.SCI.",
    "LANGUAGE AND LITERATURE":                               "LANG. & LIT.",
    "MUSIC AND BOOKS ON MUSIC":                              "MUSIC",
    "MILITARY SCIENCE":                                      "MILI.SCI.",
    "NAVAL SCIENCE":                                         "NAVAL SCI.",
    "Multidisciplinary":                                     "MULTI.",
    "Others":                                                "OTHER",
}
DISC_SHORT_INV = {v: k for k, v in DISC_SHORT.items()}



# ── Global colour palette ─────────────────────────────────────────────────────
# Consistent semantic colours used across all charts
COLORS = {
    "citing":   "#636EFA",   # blue  — Citing
    "cited":    "#EF553B",   # red   — Cited
    "self":     "#00CC96",   # teal  — Within-discipline (Self) citation
    "cross":    "#AB63FA",   # purple — Cross-discipline citation
    "incoming": "#19D3F3",   # cyan  — Incoming flow
    "internal": "#FF6692",   # pink  — Internal flow
    "outgoing": "#FFA15A",   # orange — Outgoing flow
}

FLOW_COLORS = {
    "Incoming": COLORS["incoming"],
    "Internal": COLORS["internal"],
    "Outgoing": COLORS["outgoing"],
}

# Load subject profile data
df_sub = []
for inst, path in subject_profiles.items():
    df = pd.read_csv(path, encoding="utf-8")
    df["institution"] = inst
    df_sub.append(df)
df_prof = pd.concat(df_sub, ignore_index=True)

# Detect column names from the subject profile
discipline_col = next(
    (c for c in df_prof.columns if any(k in c.lower() for k in ["discipline", "subject", "loc_label", "label"])),
    df_prof.columns[0],
)
citing_col = next((c for c in df_prof.columns if "citing" in c.lower() and "count" in c.lower()), None)
cited_col  = next((c for c in df_prof.columns if "cited"  in c.lower() and "count" in c.lower()), None)
if citing_col is None:
    citing_col = next((c for c in df_prof.columns if "citing" in c.lower()), None)
if cited_col is None:
    cited_col  = next((c for c in df_prof.columns if "cited"  in c.lower()), None)
inst_col = next(
    (c for c in df_prof.columns if any(k in c.lower() for k in ["institution", "univ", "org", "affil", "inst"])),
    None,
)

print(f"Columns detected  →  discipline: '{discipline_col}' | citing: '{citing_col}' | cited: '{cited_col}' | institution: '{inst_col}'")
df_prof.head()

Columns detected  →  discipline: 'Discipline' | citing: 'citing_count' | cited: 'cited_count' | institution: 'institution'


,Discipline,cited_count,citing_count,total_count,institution
0,SCIENCE,3170898,3557051,6727949,UNIBO
1,MEDICINE,2995492,2716909,5712401,UNIBO
2,GEOGRAPHY. ANTHROPOLOGY. RECREATION,644412,982595,1627007,UNIBO
3,TECHNOLOGY,740597,844385,1584982,UNIBO
4,AGRICULTURE,540927,781457,1322384,UNIBO


## 2. Overall Disciplinary Distribution Analysis

This section examines the overall distribution of citing and cited disciplines across all six institutions, using the `[INSTITUTION]_subject_profile.csv` dataset. The goal is to identify dominant disciplines and understand the general shape of the citation landscape before looking at directional flows.

The same data is presented through the folowing complementary chart types:
- **Butterfly Chart** — directly compares volumes per discipline per institution, of publications that are citing the institution and cited by them.
- **Grouped Bar Chart** — emphasises absolute counts side by side
- **100% Stacked Bar Chart** — highlights proportional share across institutions
- **100% Stacked Bar Chart by Volume Range** — separate panels for different volume ranges; high-range (Medicine, Science), mid-range, and low-range disciplines to address skewed distribution

### Butterfly Chart

Each institution is shown as a mirrored horizontal bar chart. Bars pointing **left (blue)** represent the number of publications citing the institution's work; bars pointing **right (red)** represent the number of publications cited by the institution. Disciplines are sorted by total volume (citing + cited) in descending order.

In [2]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROWS, COLS = 2, 3

def fmt_label(v):
    """format data lables M/K/actual"""
    a = abs(v)
    if a >= 1_000_000:
        return f"{v/1_000_000:.1f}M"
    elif a >= 1_000:
        return f"{v/1_000:.1f}K"
    else:
        return f"{v:.0f}"

def fmt_label_conditional(v):
    """format data labels only if below threshold, else return empty string"""
    if abs(v) <= 10_000:
        return fmt_label(v)
    return ""
    
fig = make_subplots(
    rows=ROWS, cols=COLS,
    subplot_titles=INSTITUTIONS,
    horizontal_spacing=0.10,
    vertical_spacing=0.18,
)

show_legend = True

for idx, inst in enumerate(INSTITUTIONS):
    row = idx // COLS + 1
    col = idx % COLS  + 1

    if inst_col is None or inst not in df_prof[inst_col].values:
        fig.add_trace(go.Bar(x=[], y=[], showlegend=False), row=row, col=col)
        fig.add_annotation(
            text="No data",
            xref=f"x{idx + 1} domain", yref=f"y{idx + 1} domain",
            x=0.5, y=0.5, showarrow=False,
            font=dict(size=14, color="gray"),
            row=row, col=col,
        )
        continue

    subset = df_prof[df_prof[inst_col] == inst]
    agg = (
        subset.groupby(discipline_col)[[citing_col, cited_col]]
        .sum().reset_index()
    )
    agg["total"]      = agg[citing_col].abs() + agg[cited_col].abs()
    agg               = agg.sort_values("total", ascending=False).reset_index(drop=True)
    agg["disc_label"] = agg[discipline_col].map(lambda x: DISC_SHORT.get(x, x))
    agg["citing_neg"] = -agg[citing_col]

    max_count = int(max(agg[citing_col].max(), agg[cited_col].max(), 1))
    pad =  int(max_count * 0.05)
    tick_vals = [-max_count, -max_count // 2, 0, max_count // 2, max_count]
    tick_text = [fmt_label(abs(v)) for v in tick_vals]

    fig.add_trace(go.Bar(
        x=agg["citing_neg"], y=agg["disc_label"],
        orientation="h", name="Citing", legendgroup="citing",
        showlegend=show_legend, marker_color=COLORS["citing"],
        customdata=agg[citing_col],
        hovertemplate="%{y}<br>Citing: %{customdata:,}<extra></extra>",
        text=agg[citing_col].map(fmt_label_conditional),
        textposition="outside",
        textfont=dict(size=7),
        cliponaxis=False,
    ), row=row, col=col)

    fig.add_trace(go.Bar(
        x=agg[cited_col], y=agg["disc_label"],
        orientation="h", name="Cited", legendgroup="cited",
        showlegend=show_legend, marker_color=COLORS["cited"],
        customdata=agg[cited_col],
        hovertemplate="%{y}<br>Cited: %{customdata:,}<extra></extra>",
        text=agg[cited_col].map(fmt_label_conditional),
        textposition="outside",
        textfont=dict(size=7),
        cliponaxis=False,
    ), row=row, col=col)

    show_legend = False

    fig.update_xaxes(
        tickmode="array", tickvals=tick_vals, ticktext=tick_text,
        range=[-max_count - pad, max_count + pad],
        row=row, col=col, tickfont=dict(size=9),
    )
    fig.update_yaxes(autorange="reversed", row=row, col=col)

fig.update_layout(
    title_text="Volume of Disciplines Citing an Institution vs Cited by an Institution",
    barmode="relative",
    height=900,
    legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="right", x=1.0),
    margin=dict(l=160, r=40, t=100, b=40),
    template="plotly_white",
)
fig.update_yaxes(tickfont=dict(size=9), automargin=True)
fig.show()

**Findings**

Across every institution, Science and Medicine dominate the citation structure. All other disciplines are much smaller. However, the balance between Science and Medicine differs strongly by institution:

UNIMI is clearly Medicine-dominant.

- UNIMI’s citation profile is highly concentrated in Medicine and Science, especially Medicine. Medicine is not only large; it is especially strong as a cited field. The gap between the top two disciplines and the rest is very large.

SNS is clearly Science-dominant.

- SNS is structurally the most distinctive institution. Science overwhelmingly dominates. It is far larger than every other discipline, and the cited Science volume is higher than the citing Science volume. This means SNS is especially visible as a cited Science institution. Medicine is a distant second. Unlike UNIMI, UNITO, or UPO, Medicine does not form an equal partner with Science. Instead, SNS has a strongly Science-centred profile.

UPO is almost evenly balanced between Science and Medicine.

- UPO is much smaller in absolute scale than UNIBO, UNITO, UNIMI, and UNIPD, but it has a very concentrated structure. Science and Medicine are almost equal and dominate the institution. After Science and Medicine, there is a steep drop. It has limited mid-disciplinary spread, with Technology as the most visible secondary area.

UNIBO and UNIPD are Science-led but relatively broad. 

- A major feature of UNIBO is its broad mid-disciplinary base. Science is the largest field, but many mid-level disciplines are active, mostly as citing fields rather than cited fields. Geo./Anthrop., Technology, Agriculture, Social Sciences, Education, Political Science, and Phil./Psych./Rel. all have visible volumes.

- Like UNIBO, UNIPD has a broad set of secondary disciplines. Several of these mid-level fields have higher citing than cited volumes, especially Geo./Anthrop., Phil./Psych./Rel., Social Sciences, Education, and Political Science.

UNITO is balanced between Science and Medicine, with Agriculture as a notable secondary field.

- UNITO combines a Science–Medicine core with a distinctive Agriculture component. Medicine is more prominent on the cited side, while Agriculture and several mid disciplines are more active on the citing side.

Cross-disciplinary pattern

A repeated pattern is that mid-level disciplines often have larger citing than cited volumes. This is visible in fields such as Geo./Anthrop., Social Sciences, Education, Political Science, Phil./Psych./Rel., and Agriculture in several institutions.

This suggests that these disciplines often contribute more as citation users than as citation recipients within the institutional profiles. By contrast, Medicine and Science, especially Medicine at UNIMI and Science at SNS, tend to be the strongest cited fields.






### Grouped Bar Chart

The same data as the Butterfly Chart, displayed as grouped vertical bars. This view makes it easier to compare absolute counts between Citing (blue) and Cited (red) for each discipline within a single institution.

In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROWS, COLS = 2, 3

fig = make_subplots(
    rows=ROWS, cols=COLS,
    subplot_titles=INSTITUTIONS,
    horizontal_spacing=0.08,
    vertical_spacing=0.22,
)

show_legend = True


for idx, inst in enumerate(INSTITUTIONS):
    row = idx // COLS + 1
    col = idx % COLS  + 1

    if inst_col is None or inst not in df_prof[inst_col].values:
        fig.add_trace(go.Bar(x=[], y=[], showlegend=False), row=row, col=col)
        fig.add_annotation(
            text="No data",
            xref=f"x{idx + 1} domain", yref=f"y{idx + 1} domain",
            x=0.5, y=0.5, showarrow=False,
            font=dict(size=14, color="gray"),
        )
        continue

    subset = df_prof[df_prof[inst_col] == inst]
    agg = (
        subset.groupby(discipline_col)[[citing_col, cited_col]]
        .sum().reset_index()
    )
    agg["total"]      = agg[citing_col] + agg[cited_col]
    agg               = agg.sort_values("total", ascending=False).reset_index(drop=True)
    agg["disc_label"] = agg[discipline_col].map(lambda x: DISC_SHORT.get(x, x))

    fig.add_trace(go.Bar(
        x=agg["disc_label"], y=agg[citing_col],
        name="Citing", legendgroup="citing",
        showlegend=show_legend, marker_color=COLORS["citing"],
        hovertemplate="%{x}<br>Citing: %{y:,}<extra></extra>",
    ), row=row, col=col)

    fig.add_trace(go.Bar(
        x=agg["disc_label"], y=agg[cited_col],
        name="Cited", legendgroup="cited",
        showlegend=show_legend, marker_color=COLORS["cited"],
        hovertemplate="%{x}<br>Cited: %{y:,}<extra></extra>",
    ), row=row, col=col)

    show_legend = False

    fig.update_xaxes(tickangle=-45, tickfont=dict(size=8), row=row, col=col)
    fig.update_yaxes(title_text="Count" if col == 1 else "", row=row, col=col)

fig.update_layout(
    title_text="Volume of Disciplines Citing an Institution vs Cited by an Institution",
    barmode="group",
    height=900,
    legend=dict(orientation="h", yanchor="bottom", y=1.03, xanchor="right", x=1),
    margin=dict(l=60, r=40, t=100, b=40),
    template="plotly_white",
)
fig.show()

**Findings**
- Across all institutions, the number of publications *citing* each institution tends to exceed the number *cited by* it (the blue bar is generally taller than the red bar).

### 100% Stacked Bar Chart

This chart normalises each institution to 100% to reveal the *proportional share* of each discipline, independent of institution size. The upper panel shows Citing share; the lower panel shows Cited share.

In [4]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.colors
from plotly.subplots import make_subplots
import pandas as pd

df = df_prof.copy()
df[discipline_col] = df[discipline_col].map(lambda x: DISC_SHORT.get(x, x))

agg = (
    df.groupby([inst_col, discipline_col])[[citing_col, cited_col]]
    .sum().reset_index()
)
for metric_col in [citing_col, cited_col]:
    totals = agg.groupby(inst_col)[metric_col].transform("sum")
    agg[f"{metric_col}_pct"] = agg[metric_col] / totals * 100

disc_order = (
    agg.groupby(discipline_col)[f"{citing_col}_pct"]
    .mean().sort_values(ascending=False).index.tolist()
)

sampled_colors = plotly.colors.sample_colorscale(
    "Plasma", [i / max(len(disc_order) - 1, 1) for i in range(len(disc_order))]
)
color_map = {d: sampled_colors[i] for i, d in enumerate(disc_order)}

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=["Citing (100%)", "Cited (100%)"],
    vertical_spacing=0.12,
)

legend_added = set()

for row_idx, metric_col in enumerate([citing_col, cited_col], start=1):
    pct_col = f"{metric_col}_pct"
    for disc in disc_order:
        sub = agg[agg[discipline_col] == disc].set_index(inst_col)
        y_vals = INSTITUTIONS
        x_vals = [sub.loc[inst, pct_col]   if inst in sub.index else 0.0 for inst in INSTITUTIONS]
        w_vals = [sub.loc[inst, metric_col] if inst in sub.index else 0   for inst in INSTITUTIONS]
        hover_vals = [
            f"<b>{inst}</b><br>{disc}<br>{pct:.1f}% ({int(w):,})"
            for inst, pct, w in zip(y_vals, x_vals, w_vals)
        ]
        show_legend = disc not in legend_added and row_idx == 1
        if row_idx == 1:
            legend_added.add(disc)

        fig.add_trace(go.Bar(
            x=x_vals, y=y_vals, orientation="h",
            name=disc, legendgroup=disc,
            showlegend=show_legend, marker_color=color_map[disc],
            customdata=hover_vals,
            hovertemplate="%{customdata}<extra></extra>",
            text=[f"{v:.1f}%" if v >= 4 else "" for v in x_vals],
            textposition="inside", textfont=dict(size=8),
        ), row=row_idx, col=1)

fig.update_layout(
    title_text="Discipline Share by Institution (100% Stacked)",
    barmode="stack",
    height=700,
    margin=dict(l=80, r=40, t=80, b=40),
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(
        orientation="v", x=1.02, xanchor="left",
        y=1.0, yanchor="top", font=dict(size=9), tracegroupgap=2,
    ),
    bargap=0.25,
)
fig.update_xaxes(range=[0, 100], ticksuffix="%", tickfont=dict(size=9), gridcolor="lightgrey")
fig.update_yaxes(tickfont=dict(size=10), autorange="reversed")
fig.show()

**Findings**

The strongest pattern is that the cited profile is more concentrated in Science and Medicine than the citing profile for every institution. In other words, when looking at what is cited, Science and Medicine take a larger share than they do on the citing side.

So, across the system, Science and Medicine become more dominant on the cited side, while mid- and low-range disciplines lose proportional weight.

UNIBO

- UNIBO has a relatively diversified citing profile.
- On the citing side, Science is 30.6% and Medicine is 23.3%, together making up 53.9%. This is one of the less concentrated Science–Medicine profiles among the institutions.
- On the cited side, however, Medicine increases strongly from 23.3% to 30.0%, while Science rises slightly from 30.6% to 31.7%. This means UNIBO’s cited profile is more concentrated in the two dominant fields, especially Medicine.

UNIBO is broad on the citing side, but its cited profile shifts toward a stronger Science–Medicine core.

UNITO

- UNITO is already fairly concentrated in Science and Medicine on the citing side: Science is 31.7% and Medicine is 28.3%, giving a combined share of 60.0%.
- The important change is on the cited side. Medicine rises sharply to 36.1%, becoming the largest discipline, while Science is 33.1%. Together, they reach 69.2%, the largest increase among all institutions.
- Agriculture is notable on the citing side at 7.5%, equal to Geo./Anthrop. at 7.5%, but these mid-level disciplines lose relative importance on the cited side.

UNITO’s cited profile is much more Medicine-oriented than its citing profile. The institution moves from a relatively balanced Science–Medicine structure to a clearly Medicine-led cited profile.

UNIMI

- UNIMI is the most Medicine-centred institution in this chart.
- On the citing side, Medicine already leads with 36.0%, ahead of Science at 32.1%. Together they account for 68.1%.
- On the cited side, Medicine becomes even more dominant, rising to 42.6%, while Science is 32.8%. The combined Science–Medicine share reaches 75.4%.
- This is one of the clearest concentration patterns in the whole chart.

UNIMI has a strongly Medicine-heavy profile, especially on the cited side. Its institutional identity is much more Medicine-based than balanced across disciplines.

UNIPD

- UNIPD appears relatively broad and balanced compared with UNIMI and UPO.
- On the citing side, Science is 31.4% and Medicine is 26.0%, together 57.4%.
- On the cited side, Science rises to 33.6% and Medicine to 29.5%, giving a combined 63.1%. Technology remains relatively important at 6.6%, while other mid-level disciplines become slightly smaller proportionally.

UNIPD is a broad generalist institution, but even here the cited side becomes more concentrated in Science and Medicine.

UPO

- UPO is highly concentrated in Science and Medicine.
- On the citing side, Science is 36.0% and Medicine is 35.2%, almost perfectly balanced. Together they account for 71.2%, already a very high concentration.
- On the cited side, the concentration increases further: Science is 39.2% and Medicine is 39.5%, together 78.7%. This makes UPO one of the most Science–Medicine-concentrated institutions in the plot.

UPO has a compact and highly concentrated profile, almost entirely structured around Science and Medicine.

SNS

- SNS is structurally different from the others because its high-discipline profile is dominated by Science, not Medicine.
- On the citing side, Science alone accounts for 49.8%, while Medicine is only 11.4%. On the cited side, Science rises further to 55.8%, while Medicine remains almost unchanged at 11.2%. 
- This means SNS is not Science–Medicine balanced like UPO, nor Medicine-heavy like UNIMI. It is overwhelmingly Science-centred.

SNS is the clearest Science-specialised institution. Its cited profile is even more Science-dominated than its citing profile.

The plot shows three main institutional profiles:

Medicine-heavy: UNIMI is the clearest case, followed by UNITO on the cited side.

Science-heavy: SNS is the strongest example, with Science alone exceeding half of the cited profile.

Broad generalist: UNIBO and UNIPD have more diversified profiles, especially on the citing side, with stronger representation across mid-level disciplines.

Highly concentrated Science–Medicine profile: UPO is the clearest case, with nearly 79% of its cited profile concentrated in Science and Medicine.

Overall, this plot shows that disciplinary diversity is greater on the citing side, while the cited side is more concentrated in the dominant high-impact fields, especially Science and Medicine.

### 100% Stacked Bar Chart by Volume Range  

Because the distribution is heavily skewed, combining all disciplines in a single chart obscures the middle and lower-ranged disciplines. From the previous results, we split disciplines into the three ranges (High / Mid / Low), normalising each range to 100% independently. This allows meaningful comparison of proportions *within* each range across institutions.


| Range | Disciplines | Rationale |
|---|---|---|
| **High** | Science, Medicine | Dominant in all institutions; the upper outliers |
| **Middle** | Geography.Anthropology.Recreation, Agriculture, Social Sciences, Education, Political Science, Philosophy.Psychology.Religion, Language and Literature, Multidisciplinary, Others | The bulk of the distribution |
| **Low** | Fine Arts, Naval Science, Military Science, General Works, Bibliography.Library Science.Information Resources, Music and Books on Music, Law, Auxiliary Sciences of History | Lower outliers with nnear-zero citation activity; invisible in combined views |




In [5]:
# Discipline range classification
HIGH_DISCS = {"MEDICINE", "SCIENCE"}
LOW_DISCS  = {
    "NAVAL SCIENCE",
    "MILITARY SCIENCE",
    "GENERAL WORKS",
    "BIBLIOGRAPHY. LIBRARY SCIENCE. INFORMATION RESOURCES",
    "MUSIC AND BOOKS ON MUSIC",
    "LAW",
    "FINE ARTS",
    "AUXILIARY SCIENCES OF HISTORY",
}

def assign_range(disc):
    if disc in HIGH_DISCS:   return "High"
    elif disc in LOW_DISCS:  return "Low"
    else:                    return "Mid"

def get_range_for_label(label):
    original = DISC_SHORT_INV.get(label, label)
    return assign_range(original)

RANGE_ORDER = ["High", "Mid", "Low"]

In [6]:
# Stacked bar chart in range groups
 
import plotly.graph_objects as go
import plotly.colors
from plotly.subplots import make_subplots

df = df_prof.copy()
df["range_group"] = df[discipline_col].map(assign_range)
df[discipline_col] = df[discipline_col].map(lambda x: DISC_SHORT.get(x, x))

agg = (
    df.groupby([inst_col, discipline_col, "range_group"])[[citing_col, cited_col]]
    .sum().reset_index()
)

def make_disc_order(range_label):
    sub = agg[agg["range_group"] == range_label].copy()
    totals = sub.groupby(inst_col)[citing_col].transform("sum")
    sub["pct"] = sub[citing_col] / totals.values * 100
    return (
        sub.groupby(discipline_col)["pct"]
        .mean().sort_values(ascending=False).index.tolist()
    )

disc_orders = {r: make_disc_order(r) for r in RANGE_ORDER}

# ── Single Plasma gradient across ALL disciplines ─────────────────────────────
# Disciplines ordered by overall citation volume (descending) so the gradient
# is consistent with the legend in other charts.
_all_discs_ordered = (
    df.groupby(discipline_col)[citing_col].sum()
    .sort_values(ascending=False).index.tolist()
)
_n  = len(_all_discs_ordered)
_pc = plotly.colors.sample_colorscale(
    "Plasma", [i / max(_n - 1, 1) for i in range(_n)]
)
GLOBAL_COLOR_MAP = {d: _pc[i] for i, d in enumerate(_all_discs_ordered)}

subplot_titles = []
for r in RANGE_ORDER:
    subplot_titles += [f"[{r}]  Citing (100%)", f"[{r}]  Cited (100%)"]

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=subplot_titles,
    vertical_spacing=0.15,
    horizontal_spacing=0.10,
)

legend_added = set()

for range_idx, range_label in enumerate(RANGE_ORDER):
    disc_order = disc_orders[range_label]
    sub_agg = agg[agg["range_group"] == range_label].copy()
    for metric_col in [citing_col, cited_col]:
        totals = sub_agg.groupby(inst_col)[metric_col].transform("sum")
        sub_agg[f"{metric_col}_pct"] = sub_agg[metric_col] / totals * 100

    row_idx = range_idx + 1

    for metric_idx, metric_col in enumerate([citing_col, cited_col]):
        col_idx = metric_idx + 1
        pct_col = f"{metric_col}_pct"

        for disc in disc_order:
            dsub = sub_agg[sub_agg[discipline_col] == disc].set_index(inst_col)
            y_vals = INSTITUTIONS
            x_vals = [dsub.loc[inst, pct_col]   if inst in dsub.index else 0.0 for inst in INSTITUTIONS]
            w_vals = [dsub.loc[inst, metric_col] if inst in dsub.index else 0   for inst in INSTITUTIONS]
            hover_vals = [
                f"<b>{inst}</b><br>{disc}<br>{pct:.1f}% ({int(w):,})"
                for inst, pct, w in zip(y_vals, x_vals, w_vals)
            ]
            legend_key  = f"{range_label}_{disc}"
            show_legend = legend_key not in legend_added and col_idx == 1
            if col_idx == 1:
                legend_added.add(legend_key)

            fig.add_trace(go.Bar(
                x=x_vals, y=y_vals, orientation="h",
                name=disc, legendgroup=legend_key,
                legendgrouptitle=dict(text=range_label) if (disc == disc_order[0] and col_idx == 1) else {},
                showlegend=show_legend,
                marker_color=GLOBAL_COLOR_MAP.get(disc, "#cccccc"),
                customdata=hover_vals,
                hovertemplate="%{customdata}<extra></extra>",
                text=[f"{v:.1f}%" if v >= 5 else "" for v in x_vals],
                textposition="inside", textfont=dict(size=8),
            ), row=row_idx, col=col_idx)

fig.update_layout(
    title_text="Discipline Share by Institution and Range (100% Stacked)",
    barmode="stack",
    height=300 * 3,
    margin=dict(l=80, r=180, t=80, b=40),
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(
        orientation="v", x=1.02, xanchor="left",
        y=1.0, yanchor="top", font=dict(size=9),
        groupclick="toggleitem", tracegroupgap=12,
    ),
    bargap=0.25,
)
fig.update_xaxes(range=[0, 100], ticksuffix="%", tickfont=dict(size=9), gridcolor="lightgrey")
fig.update_yaxes(tickfont=dict(size=10), autorange="reversed")
fig.show()

**Findings**

1. High-range disciplines: Science vs Medicine

- The main finding is that SNS is the only institution where Science overwhelmingly dominates the high range. Science accounts for more than 80% of both high-range citing and cited activity at SNS.
- By contrast, UNIMI is the most Medicine-oriented institution. Medicine accounts for 52.9% of high-range citing and rises to 56.5% of high-range cited activity.
- UNITO also shifts toward Medicine on the cited side. Its high-range citing profile is slightly Science-led, but its cited profile becomes Medicine-led. UPO is the most balanced case, with Science and Medicine almost exactly split.

At the high range, the main contrast is between Science-specialised SNS, Medicine-heavy UNIMI, and the more balanced Science–Medicine profiles of UPO, UNITO, UNIBO, and UNIPD.

2. Mid-range disciplines: more institutional differentiation

The mid-range panel is the most useful for identifying institutional specialisation outside Science and Medicine.
- UNIBO’s mid-range profile is broad, with no single field dominating completely. Technology is relatively more important on the cited side.
- UNITO has a strong mid-range identity in Agriculture and Geo./Anthrop., with Agriculture being one of its clearest secondary strengths after Science and Medicine.
- Although UNIMI is mainly Medicine-dominant in the high range, within the mid range its most distinctive field is Agriculture.
- UNIPD is broad in the mid range, but it stands out somewhat in Technology and Phil./Psych./Rel., especially compared with more Medicine-centred institutions.
- UPO’s main mid-range specialisation is Technology. This is important because UPO’s overall profile is highly concentrated in Science and Medicine, but within the mid disciplines, Technology is clearly the leading field.

At the mid range, the institutions become more differentiated: UNIBO and UNIPD are broad, UNITO and UNIMI show stronger Agriculture, UPO is more Technology-oriented, and SNS is relatively evenly spread across several mid disciplines.

3. Low-range disciplines: small overall, but internally concentrated

- The low-range panel should be interpreted carefully because these disciplines are very small in absolute volume. Still, the internal distribution shows some differences.
- Across most institutions, Fine Arts is the largest low-range discipline.
- Law is also visible in some institutions, especially UPO and SNS. SNS is distinctive because Law accounts for a very large share of the low-range category, especially compared with most other institutions.
- Low-range disciplines are generally dominated by Fine Arts, but SNS and UPO show a comparatively larger role for Law.

At the low range, Fine Arts dominates most institutions, but the results should be interpreted cautiously because the absolute volume of low-range disciplines is very small.

---

## 3. Disciplinary Citation Flow Analysis

This section examines *which disciplines cite which other disciplines* — the core of the research question. Using the `[INSTITUTION]_agg_output.csv` dataset, we treat each citing–cited discipline pair as a directed flow and adopt the perspectives:

- **Incoming:** Publications from other institutions citing the institution's work
- **Internal:** Citations between publications within the same institution
- **Outgoing:** Citations from the institution's publications to external work

The section is structured as follows:
1. **Citing–Cited Pair Ranking** — which pairs appear most frequently, per institution
2. **Self vs. Cross Citation by Discipline** — does each discipline cite within its own field or across fields?
3. **Self vs. Cross Citation by Institution × Range** — how do Self/Cross rates compare across institutions at different volume ranges?
4. **Proportion Sunburst Chart** — interactive proportional overview of the full citing→cited structure
5. **Sankey Diagram (Top N Pairs)** — directional flow and volume for a single institution
6. **Interactive Sankey Diagram** — fully customisable flow explorer

In [7]:
# Load aggregated citation pair data
import pandas as pd

dfs = []
for inst, path in agg_csvs.items():
    df = pd.read_csv(path, encoding="utf-8")
    df["institution"] = inst
    dfs.append(df)
df_agg = pd.concat(dfs, ignore_index=True)

# Apply short labels once here (all subsequent cells use df_agg directly)
df_agg["citing_loc_label"] = df_agg["citing_loc_label"].replace(DISC_SHORT)
df_agg["cited_loc_label"]  = df_agg["cited_loc_label"].replace(DISC_SHORT)

# Pre-compute derived columns used across multiple charts
df_agg["pair"] = df_agg["citing_loc_label"] + " → " + df_agg["cited_loc_label"]

df_agg["cite_type"] = df_agg.apply(
    lambda r: "Self" if r["citing_loc_label"] == r["cited_loc_label"] else "Cross",
    axis=1,
)
df_agg["range_group"] = df_agg["citing_loc_label"].map(get_range_for_label)

df_agg.head()

,flow,citing_loc_label,cited_loc_label,weight,institution,pair,cite_type,range_group
0,Outgoing,SCIENCE,SCIENCE,1599491,UNIBO,SCIENCE → SCIENCE,Self,High
1,Incoming,MEDICINE,MEDICINE,1552408,UNIBO,MEDICINE → MEDICINE,Self,High
2,Incoming,SCIENCE,SCIENCE,1413411,UNIBO,SCIENCE → SCIENCE,Self,High
3,Outgoing,MEDICINE,MEDICINE,1346293,UNIBO,MEDICINE → MEDICINE,Self,High
4,Outgoing,SCIENCE,MEDICINE,669365,UNIBO,SCIENCE → MEDICINE,Cross,High


### Citing–Cited Pair Ranking

This chart ranks the most frequent citing–cited discipline pairs per institution (Top 10 by default). Bars are colour-coded by flow type: Incoming (cyan), Internal (pink), Outgoing (orange). Use the filter buttons to isolate a single flow type; the x-axis rescales automatically per institution.

In [8]:
import pandas as pd
import plotly.graph_objects as go
import plotly.offline
from plotly.subplots import make_subplots

TOP_N = 10

LABELS = {None: "All", "Outgoing": "Outgoing", "Incoming": "Incoming", "Internal": "Internal"}

flows       = sorted(df_agg["flow"].unique())
filter_keys = [None] + flows


def fmt_weight(v):
    if v >= 1_000_000: return f"{v/1_000_000:.2f}M"
    elif v >= 1_000:   return f"{v/1_000:.2f}K"
    return str(int(v))

def fmt_hover(v):
    return f"{int(v):,}"

def make_tick_axis(x_max):
    if x_max >= 1_000_000:
        step = max(round(x_max / 5 / 500_000) * 500_000, 500_000)
        tickvals = list(range(0, int(x_max * 1.25) + step, step))
        ticktext = [f"{v/1_000_000:.1f}M" if v > 0 else "0" for v in tickvals]
    else:
        step = max(round(x_max / 5 / 50_000) * 50_000, 50_000)
        tickvals = list(range(0, int(x_max * 1.25) + step, step))
        ticktext = [f"{v/1_000:.0f}K" if v > 0 else "0" for v in tickvals]
    return tickvals, ticktext

def make_trace_data(inst, flow_filter=None):
    sub = df_agg[df_agg["institution"] == inst]
    if flow_filter is not None:
        sub = sub[sub["flow"] == flow_filter]
    sub = (
        sub.groupby("pair")["weight"].sum().reset_index()
        .sort_values("weight", ascending=False).head(TOP_N).reset_index(drop=True)
    )
    sub["rank"] = sub.index + 1
    return sub.sort_values("weight", ascending=True).reset_index(drop=True)

def make_x_config(inst, flow_filter):
    sub = df_agg[df_agg["institution"] == inst]
    if flow_filter is not None:
        sub = sub[sub["flow"] == flow_filter]
    if sub.empty or sub["weight"].sum() == 0:
        tickvals, ticktext = make_tick_axis(100_000)
        return {"range": [0, tickvals[-1]], "tickvals": tickvals, "ticktext": ticktext}
    top_weight = sub.groupby("pair")["weight"].sum().nlargest(TOP_N).max()
    x_max = float(top_weight) if not pd.isna(top_weight) else 100_000
    tickvals, ticktext = make_tick_axis(x_max)
    return {"range": [0, tickvals[-1]], "tickvals": tickvals, "ticktext": ticktext}

x_configs = {fk: [make_x_config(inst, fk) for inst in INSTITUTIONS] for fk in filter_keys}

fig = make_subplots(rows=3, cols=2, subplot_titles=INSTITUTIONS, horizontal_spacing=0.25, vertical_spacing=0.1)

n_inst  = len(INSTITUTIONS)
n_flows = len(flows)
N_ALL_TRACES    = n_flows * n_inst
N_SINGLE_TRACES = n_inst
total_traces    = N_ALL_TRACES + N_SINGLE_TRACES * n_flows

legend_added = set()

for ii, inst in enumerate(INSTITUTIONS):
    row = ii // 2 + 1
    col = ii %  2 + 1
    sub_all   = make_trace_data(inst, None)
    top_pairs = sub_all["pair"].tolist()
    ranks     = sub_all["rank"].tolist()
    y_vals    = [f"#{r}  {p}" for r, p in zip(ranks, top_pairs)]

    for flow in flows:
        sub_f = (
            df_agg[(df_agg["institution"] == inst) & (df_agg["flow"] == flow)]
            .groupby("pair")["weight"].sum()
            .reindex(top_pairs, fill_value=0).reset_index()
        )
        x_vals     = sub_f["weight"].tolist()
        hover_vals = [f"<b>{p}</b><br>{flow}: {fmt_hover(w)}" for p, w in zip(top_pairs, x_vals)]
        show_legend = flow not in legend_added
        if show_legend: legend_added.add(flow)

        fig.add_trace(go.Bar(
            x=x_vals, y=y_vals, orientation="h",
            name=flow, visible=True,
            textposition="inside", textfont=dict(size=8),
            hovertemplate="%{customdata}<extra></extra>",
            customdata=hover_vals,
            marker_color=FLOW_COLORS[flow],
            showlegend=show_legend, legendgroup=flow,
        ), row=row, col=col)

for fk in flows:
    for ii, inst in enumerate(INSTITUTIONS):
        row = ii // 2 + 1
        col = ii %  2 + 1
        sub = make_trace_data(inst, fk)
        if sub.empty:
            fig.add_trace(go.Bar(x=[], y=[], visible=False, name=inst, showlegend=False), row=row, col=col)
            continue
        y_vals     = [f"#{r}  {p}" for r, p in zip(sub["rank"], sub["pair"])]
        x_vals     = sub["weight"].tolist()
        text_vals  = [fmt_weight(w) for w in x_vals]
        hover_vals = [f"<b>#{r}  {p}</b><br>Weight: {fmt_hover(w)}" for r, p, w in zip(sub["rank"], sub["pair"], sub["weight"])]

        fig.add_trace(go.Bar(
            x=x_vals, y=y_vals, orientation="h",
            name=LABELS[fk], visible=False,
            text=text_vals, textposition="outside", textfont=dict(size=9),
            hovertemplate="%{customdata}<extra></extra>",
            customdata=hover_vals,
            marker_color=FLOW_COLORS[fk],
            cliponaxis=False, showlegend=False,
        ), row=row, col=col)

def make_visibility(active_fk):
    if active_fk is None:
        return [True] * N_ALL_TRACES + [False] * (N_SINGLE_TRACES * n_flows)
    vis = [False] * N_ALL_TRACES
    for fk in flows:
        vis += [True if fk == active_fk else False] * N_SINGLE_TRACES
    return vis

buttons = []
for fk in filter_keys:
    barmode = "stack" if fk is None else "relative"
    xaxis_updates = {}
    for i, inst in enumerate(INSTITUTIONS):
        cfg = x_configs[fk][i]
        axis_key = "xaxis" if i == 0 else f"xaxis{i+1}"
        xaxis_updates[f"{axis_key}.range"]    = cfg["range"]
        xaxis_updates[f"{axis_key}.tickvals"] = cfg["tickvals"]
        xaxis_updates[f"{axis_key}.ticktext"] = cfg["ticktext"]
    buttons.append(dict(
        label=LABELS[fk], method="update",
        args=[{"visible": make_visibility(fk)}, {"title.text": f"Top {TOP_N} Citing–Cited Pairs  |  Flow: {LABELS[fk]}", "barmode": barmode, **xaxis_updates}],
    ))

fig.update_layout(
    title=dict(text=f"Top {TOP_N} Citing–Cited Pairs  |  Flow: All"),
    barmode="stack",
    updatemenus=[dict(
        type="buttons", direction="right",
        x=0.5, xanchor="left", y=1.12, yanchor="top",
        buttons=buttons, showactive=True,
        bgcolor="#E8EAF6", bordercolor="#9FA8DA", font=dict(size=9),
    )],
    height=380 * 3,
    margin=dict(l=250, r=50, t=100, b=40),
    template="plotly_white", 
    bargap=0.25,
    legend=dict(orientation="h", x=0.5, xanchor="center", y=1.02, yanchor="bottom", font=dict(size=10)),
)
for i, inst in enumerate(INSTITUTIONS):
    cfg      = x_configs[None][i]
    axis_key = "xaxis" if i == 0 else f"xaxis{i+1}"
    fig.update_layout(**{axis_key: dict(range=cfg["range"], tickvals=cfg["tickvals"], ticktext=cfg["ticktext"], tickfont=dict(size=8), showgrid=True, gridwidth=1, gridcolor="rgba(200,200,200,0.2)")}, template="plotly_white",)
fig.update_yaxes(tickfont=dict(size=9), automargin=True)

fig.show()

**Findings**

The citation system is overwhelmingly structured around a Science–Medicine core.

Across almost all institutions, the largest pairs are:

1. Science → Science
2. Medicine → Medicine
3. Science → Medicine
4. Medicine → Science

This means the strongest citation flows are either within Science, within Medicine, or between Science and Medicine. Other disciplines only appear after these core pairs and usually at much smaller volumes.

Across the panels, Incoming and Outgoing flows dominate, while Internal flows are much smaller.

The internal component appears most clearly in the large self-pairs such as Science → Science and Medicine → Medicine, but even there it is only a small share of the total bar. This suggests that the major citation relationships are not mainly internal to the institution; they are primarily external exchanges.

Another important pattern is that outgoing flows are often large in the top pairs, especially in Science–Science and Medicine–Medicine relationships. This indicates that these institutions are heavily embedded in broader disciplinary citation networks rather than operating as closed internal systems.

UNIBO

- UNIBO has a broad structure, but its top citation pairs are still dominated by Science and Medicine, with Technology and Agriculture as secondary Science-linked fields. 
- Science is the central hub at UNIBO. Mid-level disciplines connect to the citation system mostly through Science rather than through Medicine or each other.

UNITO

- UNITO’s largest pair is Medicine → Medicine, not Science → Science. This confirms UNITO’s stronger Medicine orientation.
- UNITO has a Medicine-led citation structure, with Science as the second core field and Agriculture as the most distinctive secondary discipline.

UNIMI

- UNIMI has the strongest Medicine → Medicine pair in the whole chart. It is clearly the dominant citation relationship for the institution.
- UNIMI is highly Medicine-centred. Its citation system is dominated by Medicine self-flows, with Science as the main secondary partner.

UNIPD

- UNIPD is Science-led and relatively broad. Its secondary disciplines are integrated mainly through Science.
- Compared with UNIMI, UNIPD has a more balanced and diversified top-10 structure. Science leads, but Medicine, Technology, and Agriculture all play visible roles.

UPO

- UPO has a very compact Science–Medicine structure.
- UPO is strongly concentrated around Medicine and Science, with Technology as the most visible secondary connector.

SNS

- SNS is structurally different from the other institutions. The dominant pair is overwhelmingly Science → Science. No other pair comes close. This confirms SNS’s strong Science specialisation.
- SNS is not a balanced Science–Medicine institution. It is overwhelmingly Science-centred, and most other disciplines connect to the citation system through Science.


### Self vs. Cross Citation by Discipline

To examine the degree of interdisciplinarity in citation behaviour, each citing–cited pair is classified as **Self** (teal — same discipline cites itself) or **Cross** (purple — one discipline cites another). The 100% stacked bars show the proportion of Self vs. Cross citations for each discipline within each institution. Disciplines are ordered by total citation volume (largest at top).

Use the filter buttons to isolate a specific flow type (Incoming / Internal / Outgoing).

In [9]:
import pandas as pd
import plotly.graph_objects as go
import plotly.offline
from plotly.subplots import make_subplots

SELF_CROSS_COLORS = {"Self": COLORS["self"], "Cross": COLORS["cross"]}
LABELS = {None: "All", "Outgoing": "Outgoing", "Incoming": "Incoming", "Internal": "Internal"}

flows       = sorted(df_agg["flow"].unique())
filter_keys = [None] + flows

def fmt_weight(v):
    if v >= 1_000_000: return f"{v/1_000_000:.2f}M"
    elif v >= 1_000:   return f"{v/1_000:.2f}K"
    return str(int(v))

def make_self_cross_data(inst, flow_filter=None):
    sub = df_agg[df_agg["institution"] == inst]
    if flow_filter is not None:
        sub = sub[sub["flow"] == flow_filter]
    sub = sub.groupby(["citing_loc_label", "cite_type"])["weight"].sum().reset_index()
    totals   = sub.groupby("citing_loc_label")["weight"].sum().rename("total")
    sub      = sub.join(totals, on="citing_loc_label")
    sub["pct"] = sub["weight"] / sub["total"] * 100
    disc_order = totals.sort_values(ascending=True).index.tolist()
    return sub, disc_order, totals.to_dict()

fig = make_subplots(rows=3, cols=2, subplot_titles=INSTITUTIONS, horizontal_spacing=0.25, vertical_spacing=0.1)

n_inst     = len(INSTITUTIONS)
n_flows    = len(flows)
cite_types = ["Cross", "Self"]
N_PER_FK   = len(cite_types) * n_inst

legend_added = set()

for fi, fk in enumerate(filter_keys):
    for ct in cite_types:
        for ii, inst in enumerate(INSTITUTIONS):
            row = ii // 2 + 1
            col = ii %  2 + 1
            sub, disc_order, total_map = make_self_cross_data(inst, fk)
            sub_ct = sub[sub["cite_type"] == ct].set_index("citing_loc_label")

            pct_vals   = [float(sub_ct.loc[d, "pct"])    if d in sub_ct.index else 0.0 for d in disc_order]
            w_vals     = [int(sub_ct.loc[d, "weight"])   if d in sub_ct.index else 0   for d in disc_order]
            total_vals = [int(total_map.get(d, 0)) for d in disc_order]

            hover_vals = [
                f"<b>{d}</b><br>{ct}: {p:.2f}% ({w:,})<br>Total: {fmt_weight(t)} ({t:,})"
                for d, p, w, t in zip(disc_order, pct_vals, w_vals, total_vals)
            ]
            text_vals = [fmt_weight(t) for t in total_vals] if ct == "Self" else [""] * len(disc_order)

            show_legend = ct not in legend_added and fi == 0
            if show_legend: legend_added.add(ct)

            fig.add_trace(go.Bar(
                x=pct_vals, y=disc_order, orientation="h",
                name=ct, visible=(fi == 0),
                text=text_vals, textposition="outside", textfont=dict(size=8),
                cliponaxis=False,
                hovertemplate="%{customdata}<extra></extra>",
                customdata=hover_vals,
                marker_color=SELF_CROSS_COLORS[ct],
                showlegend=show_legend, legendgroup=ct,
            ), row=row, col=col)

def make_visibility(active_fi):
    return [fi == active_fi for fi in range(len(filter_keys)) for _ in range(N_PER_FK)]

buttons = [dict(
    label=LABELS[fk], method="update",
    args=[{"visible": make_visibility(fi)}, {"title.text": f"Self vs. Cross Citation by Discipline  |  Flow: {LABELS[fk]}"}],
) for fi, fk in enumerate(filter_keys)]

fig.update_layout(
    title=dict(text="Self vs. Cross Citation by Discipline  |  Flow: All"),
    barmode="stack",
    updatemenus=[dict(
        type="buttons", direction="right",
        x=0.5, xanchor="left", y=1.12, yanchor="top",
        buttons=buttons, showactive=True,
        bgcolor="#E8EAF6", bordercolor="#9FA8DA", font=dict(size=10),
    )],
    height=380 * 3,
    margin=dict(l=180, r=80, t=100, b=40),
    plot_bgcolor="white", paper_bgcolor="white",
    bargap=0.25,
    legend=dict(orientation="h", x=0.5, xanchor="center", y=1.02, yanchor="bottom", font=dict(size=10)),
)
for i in range(n_inst):
    axis_key = "xaxis" if i == 0 else f"xaxis{i+1}"
    fig.update_layout(**{axis_key: dict(
        range=[0, 115],
        tickvals=[0, 25, 50, 75, 100],
        ticktext=["0%", "25%", "50%", "75%", "100%"],
        tickfont=dict(size=8),
    )})
fig.update_yaxes(tickfont=dict(size=9), automargin=True)
plotly.offline.iplot(fig)

**Findings**

- The main pattern is that high-volume disciplines, especially Science and Medicine, have the strongest self-citation component, while smaller and more specialised disciplines are dominated by cross-disciplinary citation.
- In other words, large disciplines tend to cite and be cited within their own field more often, whereas smaller disciplines depend more heavily on links with other disciplines.

Institution-level Findings

UNIBO
- UNIBO shows strong self-citation in Science and Medicine, the two largest disciplines. Science has around 6.37M total volume and Medicine around 5.27M, and both contain a large green self-citation segment.
- Most mid-level disciplines are mainly cross-disciplinary, although they still have some self-citation. Very small fields such as Law, General Works, Music, Military Science, and Naval Science are almost entirely cross-disciplinary.

UNIBO has strong disciplinary self-reinforcement in Science and Medicine, but broader cross-disciplinary dependence in smaller fields.

UNITO
- UNITO also has large self-citation shares in Science and Medicine, with total volumes around 4.64M and 4.54M respectively. Agriculture, Geo./Anthrop., Technology, and Social Sciences are mostly cross-disciplinary, with smaller self-citation components.
- The lowest-volume fields are nearly fully cross-disciplinary, which suggests they do not form large self-contained citation ecosystems within the institution.

UNITO’s citation structure is centred on self-reinforcing Science and Medicine, while Agriculture and other mid-level fields mainly connect outward across disciplines.

UNIMI
- UNIMI has the largest total volumes in the chart, especially Medicine at 8.86M and Science at 7.41M. Medicine has a very large self-citation component, confirming that UNIMI’s Medicine profile is not only large but also internally strong.
- Science also has a substantial self-citation share, though the cross-disciplinary part remains important. Agriculture, Geo./Anthrop., Technology, and Social Sciences are mostly cross-disciplinary.

UNIMI’s Medicine discipline forms the strongest self-contained citation block among the institutions, reinforcing its Medicine-specialised profile.

UNIPD
- UNIPD is similar to UNIBO in having large Science and Medicine volumes, but Science is the largest discipline at around 6.33M, followed by Medicine at 5.45M.
- Science and Medicine both show substantial self-citation, while mid-level fields are mostly cross-disciplinary.

UNIPD combines large self-reinforcing Science and Medicine fields with a broad set of mid-level disciplines that are more cross-disciplinary in character.

UPO
- UPO has much smaller total volumes than the larger universities, but Science and Medicine still dominate, each around 1.05M–1.08M.
- Both Science and Medicine show meaningful self-citation, while Technology, Geo./Anthrop., Agriculture, Social Sciences, Political Science, Education, and Multi are mostly cross-disciplinary.

UPO has a compact Science–Medicine core, but outside that core, citation activity is largely cross-disciplinary.

SNS
- SNS is the most distinctive institution. Science dominates total volume at about 720K, far above Medicine and all other fields. 
- Science also has a very large self-citation component, showing that SNS’s Science specialisation is strongly self-contained.
- Medicine, Technology, Geo./Anthrop., Social Sciences, Political Science, Education, Multi, and Phil./Psych./Rel. are mostly cross-disciplinary. Agriculture has a relatively small volume and is almost entirely cross-disciplinary.

SNS is overwhelmingly Science-centred, and Science is the only discipline with a clearly strong self-citation structure. Most other SNS disciplines depend heavily on cross-disciplinary links.

Cross-disciplinary Interpretation

Across all institutions, the same hierarchy appears:
- Science and Medicine are the most self-contained disciplines. They have the largest total volumes and the largest self-citation shares.
- Mid-level disciplines are mostly cross-disciplinary. They interact strongly with other fields rather than forming isolated citation systems.
- Low-volume disciplines are almost entirely cross-disciplinary. Their totals are very small, so even small changes should not be over-interpreted.

### Self vs. Cross Citation by Institution × Discipline Range

This chart aggregates Self and Cross citation proportions *per institution* rather than per discipline, broken down by the three volume ranges. The four panels show: All Disciplines, High Range, Mid Range, and Low Range.

Use the filter buttons to examine a specific flow type.

In [10]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SELF_CROSS_COLORS = {"Self": COLORS["self"], "Cross": COLORS["cross"]}
RANGE_LABELS = {"All": "All Disciplines", "High": "High Range", "Mid": "Middle Range", "Low": "Low Range"}
FLOW_LABELS  = {None: "All", "Outgoing": "Outgoing", "Incoming": "Incoming", "Internal": "Internal"}

flows       = sorted(df_agg["flow"].unique().tolist())
filter_keys = [None] + flows

RANGE_ORDER_EXT = ["All", "High", "Mid", "Low"]

def make_inst_range_data(range_label, flow_filter=None):
    sub = df_agg.copy() if range_label == "All" else df_agg[df_agg["range_group"] == range_label].copy()
    if flow_filter is not None:
        sub = sub[sub["flow"] == flow_filter]
    if sub.empty:
        return pd.DataFrame(), {}
    agg    = sub.groupby(["institution", "cite_type"])["weight"].sum().reset_index()
    totals = agg.groupby("institution")["weight"].sum().rename("total")
    agg    = agg.join(totals, on="institution")
    agg["pct"] = agg["weight"] / agg["total"] * 100
    return agg, totals.to_dict()

GRID = [(1, 1, "All"), (1, 2, "High"), (2, 1, "Mid"), (2, 2, "Low")]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[RANGE_LABELS[rl] for _, _, rl in GRID],
    vertical_spacing=0.20, horizontal_spacing=0.10,
)

cite_types = ["Cross", "Self"]
N_PER_FK   = len(cite_types) * len(RANGE_ORDER_EXT)

legend_added = set()

for fi, fk in enumerate(filter_keys):
    for ct in cite_types:
        for row_idx, col_idx, range_label in GRID:
            agg, total_map = make_inst_range_data(range_label, fk)
            if agg.empty:
                fig.add_trace(go.Bar(x=[], y=[], visible=(fi == 0), showlegend=False), row=row_idx, col=col_idx)
                continue
            sub_ct = agg[agg["cite_type"] == ct].set_index("institution")
            x_vals     = [sub_ct.loc[inst, "pct"]        if inst in sub_ct.index else 0.0 for inst in INSTITUTIONS]
            w_vals     = [int(sub_ct.loc[inst, "weight"]) if inst in sub_ct.index else 0   for inst in INSTITUTIONS]
            total_vals = [int(total_map.get(inst, 0)) for inst in INSTITUTIONS]
            hover_vals = [
                f"<b>{inst}</b>  [{range_label}]<br>{ct}: {p:.1f}% ({w:,})<br>Total (range): {t:,}"
                for inst, p, w, t in zip(INSTITUTIONS, x_vals, w_vals, total_vals)
            ]
            show_legend = ct not in legend_added and fi == 0
            if fi == 0: legend_added.add(ct)
            fig.add_trace(go.Bar(
                x=x_vals, y=INSTITUTIONS, orientation="h",
                name=ct, legendgroup=ct,
                showlegend=show_legend, visible=(fi == 0),
                marker_color=SELF_CROSS_COLORS[ct],
                customdata=hover_vals,
                hovertemplate="%{customdata}<extra></extra>",
                cliponaxis=False,
            ), row=row_idx, col=col_idx)

def make_visibility(active_fi):
    vis = []
    for fi in range(len(filter_keys)):
        vis += [fi == active_fi] * N_PER_FK
    return vis

buttons = [dict(
    label=FLOW_LABELS[fk], method="update",
    args=[{"visible": make_visibility(fi)}, {"title.text": f"Self vs. Cross Citation by Institution × Discipline Range  |  Flow: {FLOW_LABELS[fk]}"}],
) for fi, fk in enumerate(filter_keys)]

fig.update_layout(
    title_text="Self vs. Cross Citation by Institution × Discipline Range  |  Flow: All",
    barmode="stack",
    height=480,
    margin=dict(l=80, r=120, t=110, b=40),
    plot_bgcolor="white", paper_bgcolor="white",
    updatemenus=[dict(
        type="buttons", direction="right",
        x=0.8, xanchor="left", y=1.23, yanchor="top",
        buttons=buttons, showactive=True,
        bgcolor="#E8EAF6", bordercolor="#9FA8DA", font=dict(size=10),
    )],
    legend=dict(orientation="h", x=0.5, xanchor="center", y=1.05, yanchor="bottom", font=dict(size=11)),
    bargap=0.3,
)
fig.update_xaxes(range=[0, 118], tickvals=[0, 25, 50, 75, 100], ticktext=["0%", "25%", "50%", "75%", "100%"], tickfont=dict(size=9), gridcolor="lightgrey")
fig.update_yaxes(tickfont=dict(size=10), autorange="reversed")
fig.show()




**Findings**
1. Across all disciplines, cross-disciplinary citation is slightly dominant

- In the All Disciplines chart, every institution has more cross-disciplinary than self-disciplinary citation, but the balance is not extreme.
- Most institutions sit around a 55–60% cross / 40–45% self structure. SNS is the closest to balance, with almost half of its citation activity occurring within the same discipline.

Overall citation activity is not isolated within disciplines; institutions are substantially connected across fields.

2. High-range disciplines are the only group where self-citation dominates

- In the High Range chart, which mainly reflects Science and Medicine, the green self-citation share is larger than the purple cross-citation share for all institutions.
- SNS stands out most strongly: its high-range activity is heavily self-disciplinary, meaning Science at SNS mostly connects with Science. Other institutions are also self-heavy in the high range, but less extremely.

Science and Medicine function as strong internal citation cores. They are large enough to sustain substantial same-discipline citation networks.

3. Middle-range disciplines are strongly cross-disciplinary

- In the Middle Range chart, cross-disciplinary citation dominates across every institution, usually around four-fifths of the total.
- This means disciplines such as Technology, Geo./Anthrop., Agriculture, Social Sciences, Education, Political Science, and Phil./Psych./Rel. mostly connect with other fields rather than staying within their own discipline.

Middle disciplines act as bridges between fields rather than closed citation systems.

4. Low-range disciplines are almost entirely cross-disciplinary

- In the Low Range panel, cross-disciplinary citation is nearly total for all institutions. The self-citation share is extremely small.
- Because low-range disciplines have very small volumes, this is expected: they do not have enough internal disciplinary mass to form strong same-field citation structures.

Low-volume fields depend almost completely on links to other disciplines.

5. SNS has the strongest self-disciplinary structure overall

SNS is the most self-oriented institution, especially in the high range. This fits with the earlier finding that SNS is highly Science-specialised. Since Science dominates SNS, and Science has strong self-citation, SNS naturally shows a higher overall self-citation share.

6. UNIBO, UNITO, UNIMI, UNIPD, and UPO are more cross-disciplinary overall

These institutions have large Science and Medicine cores, but their broader disciplinary mix produces more cross-disciplinary citation overall. Their middle and low ranges are strongly cross-disciplinary, which pulls the total profile toward cross-citation.

### Proportion Sunburst Chart

This interactive chart provides a proportional overview of the citing→cited discipline structure for each institution. The **inner ring** represents the citing discipline; the **outer ring** shows which disciplines it cites. Click any inner segment to drill down into that discipline's citing breakdown, and click again to zoom back out.

Use the institution buttons to switch between institutions (default: all institutions combined).

> *Note: The Sunburst emphasises proportional structure. For directional flow volumes, see the Sankey Diagram below.*

In [11]:
import pandas as pd
import plotly.graph_objects as go
import plotly.colors
import plotly.offline

INST_ALL = ["ALL"] + INSTITUTIONS

# ── Plasma gradient colour map (fixed order: citation volume descending) ──────
# Ordered by overall citation volume so the gradient is perceptually consistent
# across institutions and matches the legend order in the stacked bar charts.
_disc_volume_order = (
    df_agg.groupby("citing_loc_label")["weight"].sum()
    .sort_values(ascending=False).index.tolist()
)
_n = len(_disc_volume_order)
_plasma_colors = plotly.colors.sample_colorscale(
    "Plasma", [i / max(_n - 1, 1) for i in range(_n)]
)
DISC_COLOR = {d: _plasma_colors[i] for i, d in enumerate(_disc_volume_order)}

def make_sunburst_data(inst):
    sub = df_agg.copy() if inst == "ALL" else df_agg[df_agg["institution"] == inst]
    sub = sub.groupby(["citing_loc_label", "cited_loc_label"])["weight"].sum().reset_index()
    citing_totals = (
        sub.groupby("citing_loc_label")["weight"].sum().reset_index()
        .rename(columns={"citing_loc_label": "label"})
        .sort_values("weight", ascending=True).reset_index(drop=True)
    )
    total = citing_totals["weight"].sum()
    citing_order = citing_totals["label"].tolist()
    ids, labels, parents, values, customdata, colors = [], [], [], [], [], []

    ids.append("root"); labels.append(inst); parents.append("")
    values.append(0);   customdata.append(""); colors.append("rgba(0,0,0,0)")

    for _, row in citing_totals.iterrows():
        c, w = row["label"], int(row["weight"])
        pct  = w / total * 100
        ids.append(f"citing__{c}"); labels.append(c); parents.append("root")
        values.append(0)
        customdata.append(f"<b>Institution: {inst}</b><br><b>Citing: {c}</b><br>Weight: {w:,}<br>Share: {pct:.1f}%")
        colors.append(DISC_COLOR.get(c, "#cccccc"))

    for citing in citing_order:
        citing_w = int(citing_totals.loc[citing_totals["label"] == citing, "weight"].values[0])
        for _, row in sub[sub["citing_loc_label"] == citing].sort_values("weight", ascending=True).iterrows():
            cited, w = row["cited_loc_label"], int(row["weight"])
            pct_citing = w / citing_w * 100
            pct_total  = w / total * 100
            ids.append(f"cited__{citing}__{cited}"); labels.append(cited)
            parents.append(f"citing__{citing}"); values.append(w)
            customdata.append(
                f"<b>Institution: {inst}</b><br><b>Citing: {citing}</b><br><b>Cited: {cited}</b><br>"
                f"Weight: {w:,}<br>Share of Citing: {pct_citing:.1f}%<br>Share of Total: {pct_total:.1f}%"
            )
            colors.append(DISC_COLOR.get(citing, "#cccccc"))
    return dict(ids=ids, labels=labels, parents=parents, values=values, customdata=customdata, colors=colors)

fig = go.Figure()

for i, inst in enumerate(INST_ALL):
    d = make_sunburst_data(inst)
    fig.add_trace(go.Sunburst(
        ids=d["ids"], labels=d["labels"], parents=d["parents"], values=d["values"],
        customdata=d["customdata"], hovertemplate="%{customdata}<extra></extra>",
        branchvalues="remainder", rotation=90, sort=False,
        insidetextorientation="horizontal", visible=(i == 0),
        leaf=dict(opacity=0.75),
        marker=dict(line=dict(width=0.5, color="white"), colors=d["colors"]),
    ))

n_sunburst = len(INST_ALL)
n_dummy    = len(DISC_COLOR)

for disc, color in DISC_COLOR.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(size=10, color=color, symbol="square"),
        name=disc, showlegend=True, xaxis="x", yaxis="y",
    ))

buttons = []
for i, inst in enumerate(INST_ALL):
    visibility = [j == i for j in range(n_sunburst)] + [True] * n_dummy
    title_str  = f"Citing \u2192 Cited Flow  |  Institution: {'All Institutions' if inst == 'ALL' else inst}"
    buttons.append(dict(label=inst, method="update", args=[{"visible": visibility}, {"title": {"text": title_str}}]))

fig.update_layout(
    title=dict(text="Citing \u2192 Cited Flow  |  Institution: All Institutions"),
    updatemenus=[dict(
        type="buttons", direction="right",
        x=0.7, xanchor="left", y=1.15, yanchor="top",
        buttons=buttons, showactive=True,
        bgcolor="#E8EAF6", bordercolor="#9FA8DA", font=dict(size=10),
    )],
    showlegend=True,
    legend=dict(
        title=dict(text="Discipline", font=dict(size=11)),
        orientation="v", x=0.8, xanchor="left", y=0.5, yanchor="middle",
        font=dict(size=10), itemsizing="constant", bgcolor="rgba(0,0,0,0)",
    ),
    margin=dict(l=10, r=180, t=100, b=50), height=600,
    yaxis=dict(visible=False, fixedrange=True),
    xaxis=dict(visible=False, fixedrange=True),
    plot_bgcolor="rgba(0,0,0,0)",
)
plotly.offline.iplot(fig)

**Findings**
- *(to be completed)*

### Sankey Diagram — Top N Pairs

The Sankey diagram visualises the directional flow and volume of citations between disciplines for a single institution. Nodes on the **left** are citing disciplines; nodes on the **right** are cited disciplines. Link width is proportional to citation volume.

> *Note: Unlike the Sunburst (proportional overview), the Sankey makes volume differences directly visible. Set `INSTITUTION` and `TOP_N` in the settings block to explore different institutions.*

In [12]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.offline
import re

# ── Settings ─────────────────────────────────────────────────────────────────
TOP_N       = 30  # number of top pairs to display
INSTITUTION = "UNIMI"  # "UNIBO" / "UNITO" / "UNIMI" / "UNIPD" / "UPO" / "SNS" / "ALL"
# ─────────────────────────────────────────────────────────────────────────────

PALETTE = px.colors.qualitative.Plotly

df_inst     = df_agg.copy() if INSTITUTION == "ALL" else df_agg[df_agg["institution"] == INSTITUTION].copy()
flows       = sorted(df_inst["flow"].unique().tolist())
filter_keys = [None] + flows
LABELS      = {None: "All", "Outgoing": "Outgoing", "Incoming": "Incoming", "Internal": "Internal"}

def to_rgba(color_str, alpha=0.35):
    color_str = color_str.strip()
    if color_str.startswith("#"):
        h = color_str.lstrip("#")
        r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    else:
        r, g, b = map(int, re.findall(r"\d+", color_str))
    return f"rgba({r},{g},{b},{alpha})"

def calc_y_positions(labels, weight_series, pad=0.02):
    weights = [float(weight_series.get(l, 1e-6)) for l in labels]
    total_w = sum(weights)
    usable  = 1.0 - pad * (len(labels) - 1)
    positions, cursor = [], 0.0
    for w in weights:
        h = (w / total_w) * usable
        positions.append(round(cursor + h / 2, 4))
        cursor += h + pad
    return positions

def make_sankey_trace(flow_filter=None, top_n=TOP_N):
    if flow_filter is None:
        sub = df_inst.groupby(["citing_loc_label", "cited_loc_label"])["weight"].sum().reset_index()
    else:
        sub = (df_inst[df_inst["flow"] == flow_filter]
               .groupby(["citing_loc_label", "cited_loc_label"])["weight"].sum().reset_index())
    sub = sub.sort_values("weight", ascending=False).head(top_n).reset_index(drop=True)

    unique_citing = sub.groupby("citing_loc_label")["weight"].sum().sort_values(ascending=False).index.tolist()
    unique_cited  = sub.groupby("cited_loc_label")["weight"].sum().sort_values(ascending=False).index.tolist()
    n_left        = len(unique_citing)
    all_labels    = unique_citing + unique_cited

    all_disc   = list(dict.fromkeys(sub["citing_loc_label"].tolist() + sub["cited_loc_label"].tolist()))
    _nc = len(all_disc)
    _pc = plotly.colors.sample_colorscale("Plasma", [i/max(_nc-1,1) for i in range(_nc)])
    disc_color = {d: _pc[i] for i, d in enumerate(all_disc)}
    node_colors = [disc_color[l] for l in unique_citing] + [disc_color[l] for l in unique_cited]

    left_w  = sub.groupby("citing_loc_label")["weight"].sum()
    right_w = sub.groupby("cited_loc_label")["weight"].sum()
    node_x  = [0.01] * n_left + [0.99] * len(unique_cited)
    node_y  = calc_y_positions(unique_citing, left_w) + calc_y_positions(unique_cited, right_w)

    left_map  = {l: i          for i, l in enumerate(unique_citing)}
    right_map = {l: i + n_left for i, l in enumerate(unique_cited)}

    sources     = [left_map[r]  for r in sub["citing_loc_label"]]
    targets     = [right_map[r] for r in sub["cited_loc_label"]]
    values      = sub["weight"].tolist()
    link_colors = [to_rgba(disc_color[r]) for r in sub["citing_loc_label"]]

    node_hover = [f"<b>{l}</b><br>Total weight: {int(left_w.get(l, right_w.get(l, 0))):,}" for l in all_labels]
    link_hover = [
        f"{sub.iloc[i]['citing_loc_label']} → {sub.iloc[i]['cited_loc_label']}<br>Weight: {int(v):,}<br>Rank: #{i+1}"
        for i, v in enumerate(values)
    ]

    return go.Sankey(
        arrangement="freeform",
        node=dict(pad=10, thickness=20, line=dict(color="black", width=0.5),
                  label=all_labels, color=node_colors, x=node_x, y=node_y,
                  customdata=node_hover, hovertemplate="%{customdata}<extra></extra>"),
        link=dict(source=sources, target=targets, value=values, color=link_colors,
                  customdata=link_hover, hovertemplate="%{customdata}<extra></extra>"),
    )

fig = go.Figure()
for i, fk in enumerate(filter_keys):
    trace = make_sankey_trace(fk)
    trace.visible = (i == 0)
    fig.add_trace(trace)

buttons = [dict(
    label=LABELS[fk], method="update",
    args=[{"visible": [j == i for j in range(len(filter_keys))]},
          {"title": {"text": f"Disciplinary Citation Flow — [{INSTITUTION}]  |  Top {TOP_N} Pairs  |  Flow: {LABELS[fk]}"}}],
) for i, fk in enumerate(filter_keys)]

fig.update_layout(
    title=dict(text=f"Disciplinary Citation Flow — [{INSTITUTION}]  |  Top {TOP_N} Pairs  |  Flow: All", font=dict(size=15)),
    updatemenus=[dict(
        type="buttons", direction="right",
        x=0.75, xanchor="left", y=1.08, yanchor="top",
        buttons=buttons, showactive=True,
        bgcolor="#E8EAF6", bordercolor="#9FA8DA", font=dict(size=10),
    )],
    font_size=11, height=800, width=1200,
    margin=dict(l=20, r=20, t=100, b=30),
)
fig.show()

**Findings**
- *(to be completed)*

### Interactive Sankey Diagram

A fully customisable version of the Sankey diagram. Use the dropdowns to filter by institution(s), citing disciplines, cited disciplines, and flow type, then click **▶ Update Sankey** to redraw.

> *Ctrl (Windows) / Cmd (Mac) + click to select multiple items. Shift + click to select a range.*

In [13]:
import pandas as pd
import plotly.colors
import plotly.offline
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
import re

INSTITUTIONS_LIST = INSTITUTIONS[:]
PALETTE    = px.colors.qualitative.Plotly
all_flows  = sorted(df_agg["flow"].unique().tolist())
all_citing = sorted(df_agg["citing_loc_label"].unique().tolist())
all_cited  = sorted(df_agg["cited_loc_label"].unique().tolist())

def to_rgba(color_str, alpha=0.35):
    color_str = color_str.strip()
    if color_str.startswith("#"):
        h = color_str.lstrip("#")
        r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    else:
        r, g, b = map(int, re.findall(r"\d+", color_str))
    return f"rgba({r},{g},{b},{alpha})"

def calc_y_positions(labels, weight_series, pad=0.02):
    weights = [float(weight_series.get(l, 1e-6)) for l in labels]
    total_w = sum(weights)
    usable  = 1.0 - pad * (len(labels) - 1)
    positions, cursor = [], 0.0
    for w in weights:
        h = (w / total_w) * usable
        positions.append(round(cursor + h / 2, 4))
        cursor += h + pad
    return positions

def build_sankey(inst_sel, citing_sel, cited_sel, flow_sel):
    sub = df_agg.copy() if set(inst_sel) == set(INSTITUTIONS_LIST) else df_agg[df_agg["institution"].isin(inst_sel)].copy()
    if set(flow_sel) != set(all_flows):
        sub = sub[sub["flow"].isin(flow_sel)]
    sub = (
        sub[sub["citing_loc_label"].isin(citing_sel) & sub["cited_loc_label"].isin(cited_sel)]
        .groupby(["citing_loc_label", "cited_loc_label"])["weight"].sum().reset_index()
    )
    if sub.empty:
        return go.Figure().update_layout(title="No matching data", height=400)
    sub = sub.sort_values("weight", ascending=False).reset_index(drop=True)

    unique_citing = list(dict.fromkeys(sub["citing_loc_label"]))
    unique_cited  = list(dict.fromkeys(sub["cited_loc_label"]))
    n_left        = len(unique_citing)
    all_labels    = unique_citing + unique_cited
    all_disc      = list(dict.fromkeys(sub["citing_loc_label"].tolist() + sub["cited_loc_label"].tolist()))
    _nc = len(all_disc)
    _pc = plotly.colors.sample_colorscale("Plasma", [i/max(_nc-1,1) for i in range(_nc)])
    disc_color    = {d: _pc[i] for i, d in enumerate(all_disc)}
    node_colors   = [disc_color[l] for l in unique_citing] + [disc_color[l] for l in unique_cited]

    left_w  = sub.groupby("citing_loc_label")["weight"].sum()
    right_w = sub.groupby("cited_loc_label")["weight"].sum()
    node_x  = [0.01] * n_left + [0.99] * len(unique_cited)
    node_y  = calc_y_positions(unique_citing, left_w) + calc_y_positions(unique_cited, right_w)

    left_map  = {l: i          for i, l in enumerate(unique_citing)}
    right_map = {l: i + n_left for i, l in enumerate(unique_cited)}
    sources     = [left_map[r]  for r in sub["citing_loc_label"]]
    targets     = [right_map[r] for r in sub["cited_loc_label"]]
    values      = sub["weight"].tolist()
    link_colors = [to_rgba(disc_color[r]) for r in sub["citing_loc_label"]]

    node_hover = [f"<b>{l}</b><br>Total: {int(left_w.get(l, right_w.get(l, 0))):,}" for l in all_labels]
    link_hover = [
        f"{sub.iloc[i]['citing_loc_label']} → {sub.iloc[i]['cited_loc_label']}<br>Weight: {int(v):,}"
        for i, v in enumerate(values)
    ]
    inst_str = " + ".join(inst_sel) if set(inst_sel) != set(INSTITUTIONS_LIST) else "All"
    flow_str = " + ".join(flow_sel) if set(flow_sel) != set(all_flows) else "All"

    fig = go.Figure(go.Sankey(
        arrangement="freeform",
        node=dict(pad=10, thickness=20, line=dict(color="black", width=0.5),
                  label=all_labels, color=node_colors, x=node_x, y=node_y,
                  customdata=node_hover, hovertemplate="%{customdata}<extra></extra>"),
        link=dict(source=sources, target=targets, value=values, color=link_colors,
                  customdata=link_hover, hovertemplate="%{customdata}<extra></extra>"),
    ))
    fig.update_layout(
        title=dict(text=f"Disciplinary Citation Flow  |  Institution: {inst_str}  |  Flow: {flow_str}  |  Citing: {len(citing_sel)}  →  Cited: {len(cited_sel)}", font=dict(size=12)),
        font_size=11, height=700,
        margin=dict(l=20, r=20, t=80, b=20),
    )
    return fig

# ── Widgets ───────────────────────────────────────────────────────────────────
style      = {"description_width": "80px"}
btn_layout = widgets.Layout(width="auto", height="28px", margin="2px")

inst_select   = widgets.SelectMultiple(options=INSTITUTIONS_LIST, value=INSTITUTIONS_LIST, description="Institution:", rows=len(INSTITUTIONS_LIST), style=style, layout=widgets.Layout(width="80%", height="130px"))
flow_toggles  = widgets.SelectMultiple(options=all_flows, value=all_flows, description="Flow:", rows=len(all_flows), style=style, layout=widgets.Layout(width="80%", height="80px"))
citing_select = widgets.SelectMultiple(options=all_citing, value=all_citing[:3], description="Citing:", rows=len(all_citing), style=style, layout=widgets.Layout(width="80%", height="380px"))
cited_select  = widgets.SelectMultiple(options=all_cited, value=all_cited[:3], description="Cited:", rows=len(all_cited), style=style, layout=widgets.Layout(width="80%", height="380px"))

btn_inst_all    = widgets.Button(description="Select All", layout=btn_layout, button_style="info")
btn_inst_clear  = widgets.Button(description="Clear",      layout=btn_layout)
btn_flow_all    = widgets.Button(description="Select All", layout=btn_layout, button_style="info")
btn_flow_clear  = widgets.Button(description="Clear",      layout=btn_layout)
btn_cite_all    = widgets.Button(description="Select All", layout=btn_layout, button_style="info")
btn_cite_clear  = widgets.Button(description="Clear",      layout=btn_layout)
btn_cited_all   = widgets.Button(description="Select All", layout=btn_layout, button_style="info")
btn_cited_clear = widgets.Button(description="Clear",      layout=btn_layout)
btn_update      = widgets.Button(description="▶ Update Sankey", button_style="success", layout=widgets.Layout(width="200px", height="36px"))
out             = widgets.Output()

def on_update(_):
    inst_sel   = list(inst_select.value)
    citing_sel = list(citing_select.value)
    cited_sel  = list(cited_select.value)
    flow_sel   = list(flow_toggles.value)
    if not inst_sel:
        with out: out.clear_output(); print("⚠️  Please select at least one Institution")
        return
    if not citing_sel or not cited_sel:
        with out: out.clear_output(); print("⚠️  Please select at least one item from both Citing and Cited")
        return
    if not flow_sel:
        with out: out.clear_output(); print("⚠️  Please select at least one Flow")
        return
    with out: out.clear_output(wait=True); print("⏳ Updating Sankey diagram...")
    fig = build_sankey(inst_sel, citing_sel, cited_sel, flow_sel)
    with out: out.clear_output(wait=True); plotly.offline.iplot(fig)

def inst_all(_):    inst_select.value   = INSTITUTIONS_LIST
def inst_clear(_):  inst_select.value   = []
def flow_all(_):    flow_toggles.value  = all_flows
def flow_clear(_):  flow_toggles.value  = []
def cite_all(_):    citing_select.value = all_citing
def cite_clear(_):  citing_select.value = []
def cited_all(_):   cited_select.value  = all_cited
def cited_clear(_): cited_select.value  = []

btn_update.on_click(on_update)
btn_inst_all.on_click(inst_all);   btn_inst_clear.on_click(inst_clear)
btn_flow_all.on_click(flow_all);   btn_flow_clear.on_click(flow_clear)
btn_cite_all.on_click(cite_all);   btn_cite_clear.on_click(cite_clear)
btn_cited_all.on_click(cited_all); btn_cited_clear.on_click(cited_clear)

inst_box     = widgets.VBox([widgets.HTML("<b>Institution</b>"), widgets.HBox([btn_inst_all, btn_inst_clear]), inst_select],  layout=widgets.Layout(width="48%", margin="0 8px 8px 0"))
flow_box     = widgets.VBox([widgets.HTML("<b>Flow</b>"),        widgets.HBox([btn_flow_all, btn_flow_clear]), flow_toggles], layout=widgets.Layout(width="48%", margin="0 8px 8px 0"))
cite_box     = widgets.VBox([widgets.HTML("<b>Citing</b>"),      widgets.HBox([btn_cite_all, btn_cite_clear]), citing_select], layout=widgets.Layout(width="48%", margin="0 8px 8px 0"))
cited_box    = widgets.VBox([widgets.HTML("<b>Cited</b>"),       widgets.HBox([btn_cited_all, btn_cited_clear]), cited_select], layout=widgets.Layout(width="48%", margin="0 8px 8px 0"))
inst_flow_row = widgets.HBox([inst_box, flow_box], layout=widgets.Layout(width="70%", margin="0 0 8px 0"))
selector      = widgets.HBox([cite_box, cited_box], layout=widgets.Layout(width="70%", margin="0 0 8px 0"))
hint          = widgets.HTML("<span style='color:gray; font-size:12px'>💡 Ctrl(Win) / Cmd(Mac) + click to select multiple | Shift + click to select a range</span>")

display(
    widgets.HTML("<h4 style='margin:4px 0'>🔗 Sankey Filter</h4>"),
    inst_flow_row, selector, hint,
    widgets.HBox([btn_update], layout=widgets.Layout(margin="8px 0")),
    out,
)
out.clear_output(wait=True)
with out:
    fig = build_sankey(list(inst_select.value), list(citing_select.value), list(cited_select.value), list(flow_toggles.value))
    plotly.offline.iplot(fig)

HTML(value="<h4 style='margin:4px 0'>🔗 Sankey Filter</h4>")

HTML(value="<span style='color:gray; font-size:12px'>💡 Ctrl(Win) / Cmd(Mac) + click to select multiple | Shift…

Output()

---


## 4. Cross-Institutional Comparison

This section compares disciplinary citation patterns *across* institutions. Instead of examining flows between disciplines, we focus on how each institution's disciplinary profile deviates from the overall average. This helps identify which institutions are over- or under-represented in specific fields relative to their peers.

The dataset `[INSTITUTION]_subject_profile.csv` is used. For each institution, the proportion of each discipline (relative to all citations for that institution) is computed, and then expressed as a **deviation from the cross-institution average**.

### Relative Disciplinary Specialisation Heatmap

Each cell shows how much a given institution's proportion of a discipline deviates from the average across all six institutions (in percentage points). **Yellow** cells indicate under-representation; **purple** cells indicate over-representation relative to the mean. The y-axis labels include the cross-institution average proportion for reference.

Two heatmaps are shown: one for Citing counts, one for Cited counts.

In [14]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

custom_colorscale = [
    [0.0, "#B7990D"],   # under-indexing (yellow)
    [0.5, "#FFFFFF"],   # at average (white)
    [1.0, "#320E3B"],   # over-indexing (purple)
]

def plot_discipline_heatmap(metric_col, title):
    df = df_prof.copy()
    df[discipline_col] = df[discipline_col].map(lambda x: DISC_SHORT.get(x, x))
    agg = df.groupby([inst_col, discipline_col])[metric_col].sum().reset_index()
    totals = agg.groupby(inst_col)[metric_col].transform("sum")
    agg["proportion"] = agg[metric_col] / totals * 100

    pivot = (
        agg.pivot(index=discipline_col, columns=inst_col, values="proportion")
        .reindex(columns=INSTITUTIONS)
    )
    pivot["Average"] = pivot[INSTITUTIONS].mean(axis=1, skipna=True)
    deviation        = pivot[INSTITUTIONS].sub(pivot["Average"], axis=0)
    deviation["Average"] = pivot["Average"]
    deviation    = deviation.sort_values("Average", ascending=True)
    averages     = deviation["Average"].values
    disciplines  = deviation.index.tolist()
    y_labels     = [f"{d}  (Avg: {avg:.2f}%)" for d, avg in zip(disciplines, averages)]

    hover_dev  = deviation[INSTITUTIONS].values
    hover_true = hover_dev + averages[:, None]
    deviation  = deviation.drop(columns=["Average"])
    vmax       = np.nanmax(np.abs(deviation.values))

    hover_text = []
    for i, disc in enumerate(disciplines):
        row_texts = []
        for j, inst in enumerate(INSTITUTIONS):
            dev = hover_dev[i, j]
            tru = hover_true[i, j]
            dev_str = f"{dev:+.2f} pp" if not np.isnan(dev) else "N/A"
            tru_str = f"{tru:.2f}%"    if not np.isnan(tru) else "N/A"
            row_texts.append(
                f"<b>Institution:</b> {inst}<br>"
                f"<b>Discipline:</b> {disc}<br>"
                f"<b>Deviation from Avg:</b> {dev_str}<br>"
                f"<b>True Proportion:</b> {tru_str}"
            )
        hover_text.append(row_texts)

    fig = go.Figure(data=go.Heatmap(
        z=deviation.values, x=INSTITUTIONS, y=y_labels,
        text=hover_text, hovertemplate="%{text}<extra></extra>",
        colorscale=custom_colorscale, zmin=-vmax, zmax=vmax, zmid=0,
        colorbar=dict(title=dict(text="Deviation (pp)", side="right")),
        xgap=1, ygap=1,
    ))

    annotations = []
    for i, row in enumerate(deviation.values):
        for j, val in enumerate(row):
            if np.isnan(val):
                text, textcolor = "N/A", "lightgrey"
            else:
                text      = f"{val:+.2f}"
                textcolor = "white" if abs(val) > (vmax * 0.6) else "black"
            annotations.append(dict(
                x=INSTITUTIONS[j], y=y_labels[i], text=text,
                font=dict(color=textcolor, size=10), showarrow=False,
            ))

    fig.update_layout(
        title=dict(text=title, font=dict(size=16, family="Arial, sans-serif")),
        template="plotly_white",
        xaxis=dict(title="", tickfont=dict(size=12, weight="bold"), side="top"),
        yaxis=dict(title="", tickfont=dict(size=11)),
        height=max(400, 30 * len(disciplines) + 150),
        margin=dict(t=120, l=220, r=40, b=40),
        annotations=annotations,
    )
    fig.show()

plot_discipline_heatmap(
    metric_col=citing_col,
    title="Relative Disciplinary Specialisation: Citing Count by Institution",
)
plot_discipline_heatmap(
    metric_col=cited_col,
    title="Relative Disciplinary Specialisation: Cited Count by Institution",
)

**Findings**

CITING COUNT BY INSTITUTION

The strongest specialisation pattern is a contrast between SNS, UNIMI/UPO, and the larger generalist universities.

- SNS is strongly Science-specialised in its citing profile.
- UNIMI and UPO are strongly Medicine-specialised.
- UNIBO and UNIPD are broader and more balanced, with several modest positive deviations across mid-range disciplines.
- UNITO is moderately Medicine- and Agriculture-oriented, but less extreme than UNIMI or UPO.

The two biggest deviations in the whole heatmap are:

- SNS in Science: +14.55 pp
- SNS in Medicine: −15.35 pp

This means SNS cites far more than average in Science and far less than average in Medicine. It is the clearest disciplinary outlier.

Discipline-level Findings

Science
- Science has the highest average citing share: 35.26%.
- SNS is by far the most Science-oriented institution, with a deviation of +14.55 pp. This means Science occupies a much larger share of SNS’s citing activity than it does on average.
- UPO is only slightly above average in Science at +0.71 pp. All other institutions are below average:

So, apart from SNS, Science is not unusually over-represented in the citing profiles. In the large universities, Science is still large in absolute terms, but not unusually large relative to their own total citing mix.

Medicine
- Medicine has the second-highest average citing share: 26.72%.
- UNIMI and UPO are the clear Medicine-specialised institutions
- SNS is strongly under-represented in Medicine at −15.35 pp, confirming that its citing activity is not Medicine-centred. UNIBO is also below average at −3.37 pp, while UNIPD is close to average at −0.71 pp.

The main conclusion is that Medicine is the defining citing specialisation of UNIMI and UPO, while it is a major weakness in SNS’s profile.

Geo./Anthrop.
- Geo./Anthrop. has an average citing share of 6.76%.
- UNIBO is the most over-represented institution in this field at +1.68 pp, followed by UNITO +0.75 pp and UNIPD +0.72 pp.
- UPO is clearly below average at −1.97 pp, and UNIMI is also below average at −1.06 pp. SNS is almost average at −0.12 pp.

This suggests Geo./Anthrop. is more characteristic of UNIBO, UNITO, and UNIPD, but relatively weak in UPO and UNIMI.

Technology
- Technology averages 5.93% of citing activity.
- UNIBO is the strongest positive case at +1.33 pp, followed by UNIPD at +0.79 pp and SNS at +0.50 pp.
- UNIMI is below average at −1.57 pp, while UNITO is also below average at −1.03 pp. UPO is almost exactly average at −0.02 pp.

This indicates that Technology is relatively more important in the citing profiles of UNIBO and UNIPD, but less characteristic of UNIMI and UNITO.

Agriculture
- Agriculture averages 5.48%.
- UNITO has the strongest positive deviation: +1.99 pp. UNIMI is also above average at +1.41 pp, followed by UNIBO +1.24 pp and UNIPD +0.96 pp.
- UPO is below average at −1.21 pp, while SNS is strongly below average at −4.41 pp.

This is an important finding: Agriculture is a notable citing specialisation for UNITO, UNIMI, UNIBO, and UNIPD, but it is weak in UPO and especially SNS.

Social Sciences, Political Science, and Education
- These fields show a similar pattern.

SNS is relatively strong in all three:

Social Sciences: +2.12 pp
Political Science: +2.17 pp
Education: +2.18 pp

UNIBO is also moderately above average:

Social Sciences: +1.16 pp
Political Science: +0.87 pp
Education: +0.88 pp

By contrast, UPO and UNIMI are below average in these disciplines. UPO has negative deviations of −1.49 pp in Social Sciences, −1.56 pp in Political Science, and −1.61 pp in Education. UNIMI is also below average in all three.

This suggests that SNS and UNIBO have comparatively stronger citing activity in social-science-related fields, while UPO and UNIMI are more concentrated in the Science/Medicine core and less diversified into these areas.

Phil./Psych./Rel.
- Phil./Psych./Rel. averages 2.76%.
- UNIPD is the clearest positive outlier at +1.98 pp. This is one of UNIPD’s strongest distinctive specialisations in the citing profile.
- UNIBO and UNITO are also slightly above average, at +0.69 pp and +0.48 pp. UNIMI, UPO, and SNS are below average, especially SNS at −1.25 pp.

So, Phil./Psych./Rel. is a distinctive relative strength of UNIPD.

CITED COUNT BY INSTITUTION

The strongest pattern is a clear split between:

- SNS, which is overwhelmingly specialised in Science and strongly under-represented in Medicine;
- UNIMI and UPO, which are strongly specialised in Medicine;
- and UNIBO and UNIPD, which have broader cited profiles with stronger representation in several mid-range disciplines.

The two most extreme values in the chart are both for SNS:
SNS in Science: +18.12 pp
SNS in Medicine: −20.29 pp

This means SNS receives a much larger-than-average share of its citations in Science, while Medicine is dramatically under-represented.

Discipline-level Findings

Science
- Science has the largest average cited share, 37.71%.
- SNS is the only institution with a very strong positive Science deviation: +18.12 pp. UPO is also slightly above average at +1.46 pp.
- All four larger universities are below average in Science on the cited side
- This does not mean Science is small for these institutions in absolute terms. Rather, it means Science takes a smaller share of their cited profile than it does on average across institutions.

SNS is the clear Science-specialised institution. UPO is mildly Science-positive, while the large general universities are relatively less Science-concentrated on the cited side.

Medicine
- Medicine has the second-largest average cited share, 31.47%.
- UNIMI is the strongest Medicine-specialised institution, with a deviation of +11.10 pp. UPO is also strongly Medicine-specialised at +8.03 pp, and UNITO is moderately above average at +4.64 pp.
- SNS is extremely under-represented in Medicine at −20.29 pp. UNIPD and UNIBO are also below average, but much less dramatically.

Medicine defines the cited profile of UNIMI, UPO, and to a lesser extent UNITO. It is the weakest relative field for SNS.

Technology
- Technology has an average cited share of 5.35%.
- UNIBO is the strongest positive case at +2.06 pp, followed by UNIPD at +1.23 pp and SNS at +0.65 pp.
- UNIMI is clearly below average in Technology, at −2.17 pp, while UNITO and UPO are also below average.

Technology is relatively more important in the cited profiles of UNIBO and UNIPD. It is relatively weak for UNIMI.

Geo./Anthrop.
- Geo./Anthrop. averages 4.92% of cited activity.
- UNIBO is again the strongest positive case, with +1.54 pp. UNIPD is also above average at +0.84 pp, while UNITO is only slightly positive at +0.31 pp.
- UNIMI, UPO, and SNS are below average, especially UPO at −1.37 pp.

Geo./Anthrop. is relatively more visible in the cited profiles of UNIBO and UNIPD, but weaker for UPO and UNIMI.

Agriculture
- Agriculture has an average cited share of 4.14%.
- UNIBO and UNITO are both above average at +1.28 pp, followed by UNIMI +0.86 pp and UNIPD +0.74 pp.
- UPO is below average at −0.94 pp, and SNS is strongly below average at −3.21 pp.

Agriculture is a cited-side strength for UNIBO, UNITO, UNIMI, and UNIPD, but not for UPO or SNS.

Multi-disciplinary fields
- The Multi category averages 3.77%.
- SNS has the strongest positive deviation at +2.03 pp, while UNIPD is slightly above average at +0.41 pp.
- UNIBO is clearly below average at −1.34 pp, and UNITO, UNIMI, and UPO are also slightly negative.

SNS has a relatively strong cited presence in multi-disciplinary work, alongside its Science specialisation.

Social Sciences, Education, and Political Science
- These three fields show a similar pattern.
- SNS and UNIBO are relatively strong, while UNIMI and UPO are relatively weak.

SNS and UNIBO receive a higher-than-average cited share in social-science-related disciplines, while UPO and UNIMI are more concentrated away from these fields.

Phil./Psych./Rel.
- Phil./Psych./Rel. averages 1.93% of cited activity.
- UNIPD is the clearest positive outlier at +1.76 pp. UNIBO is also above average at +0.86 pp, while UNITO is mildly positive at +0.31 pp.
- UPO and SNS are below average, with −1.22 pp and −1.03 pp respectively.

Phil./Psych./Rel. is one of UNIPD’s most distinctive cited-side strengths.

---


## 5. Discussion

*(to be completed)*

(Some notes)

- Why is the share of medicine and science so high in most institutions?
  - Is it because they produce much more publications, in terms of more people practicing them, or frequency? 
  - Is it because they are a larger representative of subcategories within them? (Chemistry, Biology, Physics, Maths etc)
- Do the characteristics of the University reflect the patterns?
- Why is the share of LAW so low (it seems a pretty general discipline)
